# 08 Deep Learning — Evaluation & Comparison with Phase 1

---

## Purpose

Evaluate the two trained DL models on the **same 20 % hold-out test set** used in
Phase 1, then build a comprehensive comparison across all four models.

### Outputs
- Confusion matrices and ROC curves
- Classification reports
- **Phase 1 (ML) vs Phase 2 (DL)** comparison table and bar chart
- Error analysis: disagreement patterns, accuracy by text length, confidence distributions
- Saved metrics (JSON/CSV) and predictions for Phase 3 hybrid model

### Prerequisites
Run `06_dl_data_preprocessing.ipynb` → `07_dl_model_training.ipynb` first.

## 1. Imports and Load Artifacts

In [ ]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, auc,
)
from IPython.display import Markdown, display

warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.deep_learning import (
    LSTMConfig, TextCNNConfig,
    TextClassificationDataset,
    BiLSTMClassifier, SETextCNNClassifier,
    evaluate_lstm, evaluate_cnn,
    get_device,
)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 5)

RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
METRICS_DIR = PROJECT_ROOT / 'metrics'
for d in [RESULTS_DIR, FIGURES_DIR, METRICS_DIR]:
    d.mkdir(exist_ok=True)

device = get_device()
print(f'PyTorch {torch.__version__} | Device: {device}')

In [ ]:
# Load test data — both models share the same word-level token tensors
test_ids    = torch.load(RESULTS_DIR / 'dl_test_ids.pt', weights_only=True)
y_test      = np.load(RESULTS_DIR / 'dl_y_test.npy')
X_test_text = np.load(RESULTS_DIR / 'dl_test_texts.npy', allow_pickle=True)

with open(RESULTS_DIR / 'dl_tokenizer.json') as f:
    tok_data = json.load(f)
vocab_size = len(tok_data['word2idx'])

print(f'Test set: {len(y_test):,} samples')
print(f'Word-level tensor shape: {test_ids.shape}')
print(f'Vocab size: {vocab_size:,}')
print(f'Class balance: Lost={int((y_test==0).sum()):,}  Won={int((y_test==1).sum()):,}')

## 2. Load Trained Models

In [ ]:
# Reconstruct BiLSTM and load best checkpoint
lstm_cfg = LSTMConfig(
    vocab_size=vocab_size, embed_dim=128, hidden_dim=256,
    num_layers=2, dropout=0.3, bidirectional=True, num_classes=2,
)
lstm_model = BiLSTMClassifier(lstm_cfg, pad_idx=0)
lstm_model.load_state_dict(torch.load(RESULTS_DIR / 'model_lstm_best.pt', weights_only=True))
lstm_model = lstm_model.to(device)
lstm_model.eval()
print(f'BiLSTM loaded ({sum(p.numel() for p in lstm_model.parameters()):,} params)')

# Reconstruct SE-TextCNN and load best checkpoint
cnn_cfg = TextCNNConfig(
    vocab_size=vocab_size, embed_dim=128,
    kernel_sizes=[3, 4, 5], num_filters=128,
    se_reduction=16, dropout=0.4, num_classes=2,
)
cnn_model = SETextCNNClassifier(cnn_cfg, pad_idx=0)
cnn_model.load_state_dict(torch.load(RESULTS_DIR / 'model_cnn_best.pt', weights_only=True))
cnn_model = cnn_model.to(device)
cnn_model.eval()
print(f'SE-TextCNN loaded ({sum(p.numel() for p in cnn_model.parameters()):,} params)')

In [ ]:
# Both models share the same word-level test dataset
test_dataset = TextClassificationDataset(test_ids, y_test)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

print(f'Shared test loader: {len(test_loader)} batches of 128')
print('Both BiLSTM and SE-TextCNN use the same word-level token tensors.')

## 3. Generate Predictions

In [ ]:
criterion = nn.CrossEntropyLoss()

# BiLSTM evaluation
lstm_test_loss, lstm_test_acc, lstm_preds, lstm_probs = evaluate_lstm(
    lstm_model, test_loader, criterion, device,
)

lstm_metrics = {
    'Accuracy':  accuracy_score(y_test, lstm_preds),
    'Precision': precision_score(y_test, lstm_preds),
    'Recall':    recall_score(y_test, lstm_preds),
    'F1':        f1_score(y_test, lstm_preds),
    'ROC-AUC':   roc_auc_score(y_test, lstm_probs),
    'AP':        average_precision_score(y_test, lstm_probs),
}

print('BiLSTM + Attention — Test Metrics')
for k, v in lstm_metrics.items():
    print(f'  {k:12s}: {v:.4f}')

In [ ]:
# SE-TextCNN evaluation
cnn_test_loss, cnn_test_acc, cnn_preds, cnn_probs = evaluate_cnn(
    cnn_model, test_loader, criterion, device,
)

cnn_metrics = {
    'Accuracy':  accuracy_score(y_test, cnn_preds),
    'Precision': precision_score(y_test, cnn_preds),
    'Recall':    recall_score(y_test, cnn_preds),
    'F1':        f1_score(y_test, cnn_preds),
    'ROC-AUC':   roc_auc_score(y_test, cnn_probs),
    'AP':        average_precision_score(y_test, cnn_probs),
}

print('SE-TextCNN (TextCNN + Channel Attention) — Test Metrics')
for k, v in cnn_metrics.items():
    print(f'  {k:12s}: {v:.4f}')

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_names = ['Lost', 'Won']

for ax, preds, title in [
    (axes[0], lstm_preds, 'BiLSTM + Attention'),
    (axes[1], cnn_preds,  'SE-TextCNN'),
]:
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{title} — Confusion Matrix')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/dl_confusion_matrices.png')

## 5. Classification Reports

In [ ]:
print('='*60)
print('BiLSTM + Attention — Classification Report')
print('='*60)
print(classification_report(y_test, lstm_preds, target_names=class_names))

print('='*60)
print('SE-TextCNN (TextCNN + Channel Attention) — Classification Report')
print('='*60)
print(classification_report(y_test, cnn_preds, target_names=class_names))

## 6. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for probs, name, color in [
    (lstm_probs, 'BiLSTM + Attention', '#1f77b4'),
    (cnn_probs,  'SE-TextCNN',         '#ff7f0e'),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{name} (AUC = {roc_auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Deep Learning Models', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/dl_roc_curves.png')

## 7. Phase 1 (ML) vs Phase 2 (DL) — Direct Comparison

Phase 1 models (Logistic Regression and Random Forest) used **engineered tabular
features + precomputed embeddings → PCA (100 components)**.  The DL models here
use **raw conversation text** end-to-end.  Same test set, same stratification.

In [ ]:
# Phase 1 ML results (from metrics/evaluation_metrics.json)
phase1_metrics = {
    'Logistic Regression (ML)': {
        'Accuracy': 0.9634, 'Precision': 0.9504, 'Recall': 0.9777,
        'F1': 0.9639, 'ROC-AUC': 0.9957, 'AP': 0.9957,
    },
    'Random Forest (ML)': {
        'Accuracy': 0.9594, 'Precision': 0.9453, 'Recall': 0.9751,
        'F1': 0.9600, 'ROC-AUC': 0.9951, 'AP': 0.9951,
    },
}

all_results = {
    **phase1_metrics,
    'BiLSTM + Attention (DL)': lstm_metrics,
    'SE-TextCNN (DL)': cnn_metrics,
}

comparison_df = pd.DataFrame(all_results).T
comparison_df = comparison_df[['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'AP']]

print('\n' + '='*80)
print('FULL MODEL COMPARISON — Phase 1 (ML) vs Phase 2 (DL)')
print('='*80)
display(comparison_df.round(4).style.highlight_max(axis=0, color='lightgreen'))

In [ ]:
# Grouped bar chart
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
model_names = list(all_results.keys())
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

x = np.arange(len(metrics_to_plot))
width = 0.18

fig, ax = plt.subplots(figsize=(14, 6))
for i, (model_name, color) in enumerate(zip(model_names, colors)):
    values = [all_results[model_name][m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, values, width, label=model_name, color=color, alpha=0.85)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Phase 1 (ML) vs Phase 2 (DL) — Model Comparison', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.legend(loc='lower right', fontsize=10)
ax.set_ylim(0.85, 1.02)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_vs_dl_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/ml_vs_dl_comparison.png')

## 8. Error Analysis — Where DL Succeeds / Fails vs ML

Understanding *complementary* strengths is key for Phase 3 (hybrid model).

In [ ]:
lstm_correct = (lstm_preds == y_test)
cnn_correct  = (cnn_preds  == y_test)

both_correct     = lstm_correct & cnn_correct
both_wrong       = (~lstm_correct) & (~cnn_correct)
lstm_only_correct = lstm_correct & (~cnn_correct)
cnn_only_correct  = cnn_correct  & (~lstm_correct)

print('Test Set Disagreement Analysis (DL models)')
print(f'  Both correct:            {both_correct.sum():>6,} ({both_correct.mean():.2%})')
print(f'  Both wrong:              {both_wrong.sum():>6,} ({both_wrong.mean():.2%})')
print(f'  Only BiLSTM correct:     {lstm_only_correct.sum():>6,} ({lstm_only_correct.mean():.2%})')
print(f'  Only SE-TextCNN correct: {cnn_only_correct.sum():>6,} ({cnn_only_correct.mean():.2%})')
print(f'\nAgreement rate: {(lstm_preds == cnn_preds).mean():.2%}')

In [ ]:
# Accuracy by conversation length
test_lengths = np.array([len(t.split()) for t in X_test_text])

length_bins = [0, 100, 250, 500, 1000, float('inf')]
bin_labels = ['<100', '100-250', '250-500', '500-1000', '1000+']

lstm_acc_by_len, cnn_acc_by_len, counts = [], [], []
for lo, hi, label in zip(length_bins[:-1], length_bins[1:], bin_labels):
    mask = (test_lengths >= lo) & (test_lengths < hi)
    n = mask.sum()
    counts.append(n)
    lstm_acc_by_len.append(lstm_correct[mask].mean() if n > 0 else 0)
    cnn_acc_by_len.append(cnn_correct[mask].mean() if n > 0 else 0)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(bin_labels))
w = 0.35
ax.bar(x - w/2, lstm_acc_by_len, w, label='BiLSTM', color='#1f77b4', alpha=0.8)
ax.bar(x + w/2, cnn_acc_by_len,  w, label='SE-TextCNN', color='#ff7f0e', alpha=0.8)
ax.set_xlabel('Text Length (words)')
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy by Conversation Length')
ax.set_xticks(x)
ax.set_xticklabels([f'{l}\n(n={c:,})' for l, c in zip(bin_labels, counts)])
ax.legend()
ax.set_ylim(0.0, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_accuracy_by_length.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/dl_accuracy_by_length.png')

In [ ]:
# Confidence distributions for correct vs incorrect predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, probs, correct, title in [
    (axes[0], lstm_probs, lstm_correct, 'BiLSTM + Attention'),
    (axes[1], cnn_probs,  cnn_correct,  'SE-TextCNN'),
]:
    confidence = np.where(y_test == 1, probs, 1 - probs)
    ax.hist(confidence[correct],  bins=30, alpha=0.7, label='Correct',   color='green', density=True)
    ax.hist(confidence[~correct], bins=30, alpha=0.7, label='Incorrect', color='red',   density=True)
    ax.set_xlabel('Prediction Confidence')
    ax.set_ylabel('Density')
    ax.set_title(f'{title} — Confidence Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/dl_confidence_distribution.png')

## 9. Save All Metrics and Predictions

In [ ]:
# DL metrics
dl_metrics = {
    'BiLSTM + Attention': lstm_metrics,
    'SE-TextCNN': cnn_metrics,
}
with open(METRICS_DIR / 'dl_evaluation_metrics.json', 'w') as f:
    json.dump(dl_metrics, f, indent=2)

pd.DataFrame(dl_metrics).T.to_csv(METRICS_DIR / 'dl_evaluation_metrics.csv')

# Full ML vs DL comparison
comparison_df.to_csv(METRICS_DIR / 'ml_vs_dl_comparison.csv')
comparison_df.to_json(METRICS_DIR / 'ml_vs_dl_comparison.json')

# Classification reports
dl_reports = {
    'BiLSTM + Attention': classification_report(y_test, lstm_preds, target_names=class_names, output_dict=True),
    'SE-TextCNN': classification_report(y_test, cnn_preds, target_names=class_names, output_dict=True),
}
with open(METRICS_DIR / 'dl_classification_reports.json', 'w') as f:
    json.dump(dl_reports, f, indent=2)

# Confusion matrices
dl_cm = {
    'BiLSTM + Attention': confusion_matrix(y_test, lstm_preds).tolist(),
    'SE-TextCNN': confusion_matrix(y_test, cnn_preds).tolist(),
}
with open(METRICS_DIR / 'dl_confusion_matrices.json', 'w') as f:
    json.dump(dl_cm, f, indent=2)

print('Metrics saved to metrics/')
print('  dl_evaluation_metrics.json / .csv')
print('  ml_vs_dl_comparison.json / .csv')
print('  dl_classification_reports.json')
print('  dl_confusion_matrices.json')

## 10. Summary

In [ ]:
best_dl_name = 'BiLSTM + Attention' if lstm_metrics['F1'] >= cnn_metrics['F1'] else 'SE-TextCNN'
best_dl_f1   = max(lstm_metrics['F1'], cnn_metrics['F1'])
best_ml_f1   = 0.9639  # Phase 1 Logistic Regression

summary = f"""
## Phase 2 Deep Learning — Evaluation Summary

### Test Set Results

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|-------|----------|-----------|--------|----|---------|
| Logistic Regression (ML) | 0.9634 | 0.9504 | 0.9777 | 0.9639 | 0.9957 |
| Random Forest (ML) | 0.9594 | 0.9453 | 0.9751 | 0.9600 | 0.9951 |
| BiLSTM + Attention (DL) | {lstm_metrics['Accuracy']:.4f} | {lstm_metrics['Precision']:.4f} | {lstm_metrics['Recall']:.4f} | {lstm_metrics['F1']:.4f} | {lstm_metrics['ROC-AUC']:.4f} |
| **SE-TextCNN (DL)** | **{cnn_metrics['Accuracy']:.4f}** | **{cnn_metrics['Precision']:.4f}** | **{cnn_metrics['Recall']:.4f}** | **{cnn_metrics['F1']:.4f}** | **{cnn_metrics['ROC-AUC']:.4f}** |

### Key Findings

- **Best DL model**: {best_dl_name} (F1 = {best_dl_f1:.4f})
- **Best ML model**: Logistic Regression (F1 = {best_ml_f1:.4f})
- {'✅ DL improves over ML baseline — end-to-end text features capture signals hand-crafted features miss' if best_dl_f1 > best_ml_f1 else '⚠️ ML baseline remains competitive — strong engineered features are hard to beat on this dataset'}

### What DL brings that ML cannot

1. **End-to-end learning**: No manual feature engineering — learns directly from raw tokens
2. **N-gram pattern detection**: TextCNN kernels automatically discover predictive sales phrases
3. **Channel Attention (SE block)**: Re-weights n-gram groups per sample — fully interpretable via SE weights
4. **Speed vs complexity trade-off**: SE-TextCNN trains in ~2 min with no pretrained dependencies

### Architecture Comparison

| | BiLSTM + Attention | SE-TextCNN |
|-|--------------------|-----------|
| Mechanism | Temporal (sequence order) | Local n-gram + SE channel attention |
| Speed | ~5 min/epoch | ~1-2 min/epoch |
| Params | ~4 M | ~4 M |
| Attention type | Additive (Bahdanau) | Channel-wise (Squeeze-and-Excite) |
| Best for | Order-dependent signals | Key-phrase / n-gram signals |

### Next → Phase 3 (Hybrid)

Combine ML tabular features with DL text representations in a fusion model.
"""
display(Markdown(summary))